In [1]:
import numpy as np
import re #for data preprocessing and exploration
from collections import Counter #for calculating words frequencies
import random
from datasets import load_dataset # for loading the data, huggingface library
from sklearn.cluster import KMeans

C:\Users\stank\python_projects\word2vec_numpy\onlyNumpy\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Loading Data
Data is a set of tokens extracted from the verified articles on Wikipedia. For more info visit:
https://huggingface.co/datasets/Salesforce/wikitext

In [2]:
dataset = load_dataset("Salesforce/wikitext", name="wikitext-2-v1", split="train")
dataset = dataset["text"]

#### Data exploration

In [3]:
#showing part of the data
for i, row in enumerate(dataset):
    print(row[:200])
    if i >= 4:
        break


 = Valkyria Chronicles III = 


 Senjō no Valkyria 3 : <unk> Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playin
 The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adju


From eye glance at the data we can see that data probably contains huge number of special characters. They are not really important for the model and should be filtered out. With the code below we can confirm this hypothesis

In [4]:
#function returns true if word contain any character...
#that is not letter or number
def has_special_chars(word):
    return bool(re.search(r'[^a-zA-Z0-9]', word))

special_counter = Counter()
for i, row in enumerate(dataset):
    words = row.split() #spliting rows into words, by default on space characters
    for w in words:
        if has_special_chars(w):
            special_counter[w] += 1
    if i >= 10000: #we just look through first 10000 rows. It is enough for exploration
        break

print("Most common words with special characters:")
print(special_counter.most_common(20))

Most common words with special characters:
[(',', 27365), ('.', 19705), ('<unk>', 15189), ('=', 7933), ('"', 7683), ('@-@', 4825), ("'s", 3596), (')', 3076), ('(', 3075), (';', 1361), ('–', 1145), (':', 994), ('@.@', 930), ('@,@', 784), ("'", 737), ('—', 320), ('/', 301), ('%', 267), ('$', 262), ('[', 207)]


The number of punctuations is indeed quite big. Thankfully they exist in the dataset as a separate words, so we can filter them out without any trouble. We can also check for any long words that are unusual

In [5]:
long_words = []

for i, row in enumerate(dataset):
    for word in row.split():
        if len(word) > 20:
            long_words.append(w)
    if i >= 10000:
        break

print("Some very long words:")
print(long_words[:20])

Some very long words:
[]


There are no any unusually long words. 

#### Preprocessing
Here is the preprocess function for dividing dataset into word tokens with filtering special character words out. Capitalisation is not important for the word meaning, so we should turn all words to lowercase

In [6]:
def preprocess(dataset):
    for text in dataset:
        text = text.lower()
        tokens = re.findall(r'\w+', text) #filtering special characters out
        words.extend(tokens)
    return words

In [7]:
words = preprocess(dataset) #list of all words ["the", "fox", "the", "dog"]
vocab = list(set(words)) #set of words - duplicates are removed ["the", "fox", "dog"]
vocab_size = len(vocab) #number of words in the set
print(words[:20])
print(vocab[:20])
print(len(words))
print(vocab_size)

['valkyria', 'chronicles', 'iii', 'senjō', 'no', 'valkyria', '3', 'unk', 'chronicles', 'japanese', '戦場のヴァルキュリア3', 'lit', 'valkyria', 'of', 'the', 'battlefield', '3', 'commonly', 'referred', 'to']
['magna', 'matt', 'gibraltar', 'gunners', 'orchestration', 'stance', 'encounters', 'northern', 'crucial', 'nayavāda', 'menacing', 'rhetoric', 'pocket', 'aerobatics', 'plank', 'bottleneck', 'authoritative', 'apostolic', 'plated', '1686']
1750345
28710


#### Further words exploration
To make the model better, we should focus only on most popular words. Therefore good thing is to check number of least popular words

In [8]:
word_counts = Counter(words)
sorted_words = word_counts.most_common()
sorted_words.reverse()

print("Least popular words:")
for i in range(10):
    word, count = sorted_words[i]
    print("Top", i+1, "least popular word:", word, "with count=", count)

sorted_words.reverse()
print("Most popular words:")
for i in range(10):
    word, count = sorted_words[i]
    print("Top", i+1, "most popular word:", word, "with count=", count)

Least popular words:
Top 1 least popular word: gallinae with count= 3
Top 2 least popular word: intergrades with count= 3
Top 3 least popular word: northeasterly with count= 3
Top 4 least popular word: tuscola with count= 3
Top 5 least popular word: roundabouts with count= 3
Top 6 least popular word: zoromski with count= 3
Top 7 least popular word: forrester with count= 3
Top 8 least popular word: kreutzer with count= 3
Top 9 least popular word: prefaced with count= 3
Top 10 least popular word: philipp with count= 3
Most popular words:
Top 1 most popular word: the with count= 130768
Top 2 most popular word: of with count= 57030
Top 3 most popular word: unk with count= 54625
Top 4 most popular word: and with count= 50735
Top 5 most popular word: in with count= 45015
Top 6 most popular word: to with count= 39521
Top 7 most popular word: a with count= 36523
Top 8 most popular word: was with count= 21008
Top 9 most popular word: on with count= 15140
Top 10 most popular word: as with count=

In [9]:
def word_number_checker(words, target_number):
    words_number = 0
    sorted_words = Counter(words).most_common()
    for word, count in sorted_words:
        if count>=target_number:
            words_number+=1
    print("Appearing more than", target_number, "times:", words_number)

word_number_checker(words, 1)
word_number_checker(words, 5)
word_number_checker(words, 10)
word_number_checker(words, 20)

Appearing more than 1 times: 28710
Appearing more than 5 times: 20352
Appearing more than 10 times: 12884
Appearing more than 20 times: 8093


#### Further preprocessing

To make the training reasonably fast I'm going to use only first 500k word tokens for model training. This is obviously optional, I do this because primary purpose of this model is to be quick to train word2vec example 

In [10]:
#THIS CODE IS OPTIONAL FOR FASTER MODEL TRAINING
words = words[:500000]
vocab = list(set(words)) #Some words might not be present in the first 500k 
vocab_size = len(vocab)
word_number_checker(words, 1)
word_number_checker(words, 3)
word_number_checker(words, 5)

Appearing more than 1 times: 20878
Appearing more than 3 times: 13570
Appearing more than 5 times: 9307


More than 8000 from 28000 distinct word appear less than 5 times. Model could benefit from not includint them in input and focusing only on neighborhood of popular words. (for the smaller dataset version I'm going to filter out words that appear less than 3 times - 8k out of 21k)

In [11]:
min_count = 3
word_counts = Counter(words)
vocab = [word for word, c in word_counts.items() if c >= min_count]
vocab_set = set(vocab) #Convert to set to avoid going through list...
#several times in a for loop
words = [word for word in words if word in vocab_set]
vocab_size = len(vocab)

print(vocab[:20])
print(vocab_size)
print(len(words))

['valkyria', 'chronicles', 'iii', 'senjō', 'no', '3', 'unk', 'japanese', '戦場のヴァルキュリア3', 'lit', 'of', 'the', 'battlefield', 'commonly', 'referred', 'to', 'as', 'outside', 'japan', 'is']
13570
489575


#### Model preparation
I convert words from input to numeric values and generate pairs of neighbouring words. Important parameter to set is window size, which determines how far word neighborhood reaches.

In [12]:
word2idx = {w:i for i,w in enumerate(vocab)} #Map of words to number {'the': 0,'fox': 1} 
idx2word = {i:w for w,i in word2idx.items()} #the same but other way around
print(list(word2idx.items())[:20])

[('valkyria', 0), ('chronicles', 1), ('iii', 2), ('senjō', 3), ('no', 4), ('3', 5), ('unk', 6), ('japanese', 7), ('戦場のヴァルキュリア3', 8), ('lit', 9), ('of', 10), ('the', 11), ('battlefield', 12), ('commonly', 13), ('referred', 14), ('to', 15), ('as', 16), ('outside', 17), ('japan', 18), ('is', 19)]


In [13]:
corpus = [word2idx[w] for w in words] 
#words in sentence are replaced by numbers mapped by word2idx 
#ex. corpus for 'the fox' is [0, 1]
print(corpus[:20])

[0, 1, 2, 3, 4, 0, 5, 6, 1, 7, 8, 9, 0, 10, 11, 12, 5, 13, 14, 15]


In [14]:
#Function for mapping pairs of neighbouring words
#ex for window_size=1, corpus=[0,1,2] output would be:
#[(0,1), (1,0), (1,2), (2,1)]
#neighbourhouds are repeating ex. in sentence 'the fox' 2 facts of neighborhood...
#are recorded: the is neighbour of fox and fox is neighbour of the.
#window size is maximum distance of recorded neighborhood
def generate_pairs(corpus, window_size=2): 
    pairs = []
    for i, word in enumerate(corpus): #for every word in sentence
        for j in range(-window_size, window_size+1): #for every its neighbour
            #word do not record itselfas neighbour
            if j==0:
                continue
            neighbour_pos = i+j
            if 0<=neighbour_pos<len(corpus):
                neighbour = corpus[neighbour_pos]
                pairs.append((word, neighbour))
    return pairs

In [15]:
pairs = generate_pairs(corpus)
print(pairs[:20])

[(0, 1), (0, 2), (1, 0), (1, 2), (1, 3), (2, 0), (2, 1), (2, 3), (2, 4), (3, 1), (3, 2), (3, 4), (3, 0), (4, 2), (4, 3), (4, 0), (4, 5), (0, 3), (0, 4), (0, 5)]


#### Model Training
The word2vec model implemented here is skip gram with negative sampling. It means that the model predicts context words based on target word and that training is improved by comparing words not only with its' neighbours, but also with random words from the set

In [17]:
class Word2vec:
    def __init__(self, vocab_size, embedding_dim=20, learning_rate=0.05,
                 negative_samples=2, epochs=5):
        self.vocab_size = vocab_size
        #Model parameters
        self.embedding_dim = embedding_dim #number of dimensions(fetures) for each word
        self.learning_rate = learning_rate
        #number of words used in negative sampling for every iteration
        self.negative_samples = negative_samples
        self.epochs = epochs
        
        #Each word has 2 dimension vectors:
        #W_in - vector of word being embedded as target in some situation
        #W_out - vector of word being embedded as context in some situation
        #At the end similarity of words can be checked based on similarity...
        #of one of the vectors
        self.W_in = np.random.randn(vocab_size, embedding_dim) * 0.01
        self.W_out = np.random.randn(vocab_size, embedding_dim) * 0.01

    def train(self, pairs):
        for epoch in range(self.epochs):
            loss = 0
            for word, neighbour in pairs:
                v_target = self.W_in[word].copy()
                v_context = self.W_out[neighbour].copy()
                #forward pass - vectors multiplication... 
                #if related close to 1, if no close to 0
                score = np.dot(v_target, v_context) 
                prob = self.sigmoid(score)

                loss += -np.log(prob + 1e-10)
                grad = prob-1
                self.W_in[word] -= self.learning_rate * grad * v_context
                self.W_out[neighbour] -= self.learning_rate * grad * v_target
                #logic behind backpropagation: 
                #adding neighbour vector to the target would make vectors the same:
                #W_in+v_context=W_out+v_target
                #Obviously we don't want to make them the same after...
                #just one appearance near each other, so we use learning rate...
                #to smooth the effect.
                #The backpropagation should be the more boosted the more words...
                #are not related yet, gradient is higher if probabilty of words...
                #being related is close to 0.
                #Minus sign is add to cancel effect of negative gradient

                #Negative sampling works the same way only the gradient being...
                #positive instead of negative. It is used to decrease connection..
                #of word to a random word. 
                for i in range(self.negative_samples):
                    neg_word = random.randint(0,self.vocab_size-1)
                    v_target = self.W_in[word].copy()
                    v_negative = self.W_out[neg_word].copy()
                    score = np.dot(v_target, v_negative)
                    prob = self.sigmoid(score)

                    loss += -np.log(1 - prob + 1e-10)
                    grad = prob
                    self.W_in[word] -= self.learning_rate * grad * v_negative
                    self.W_out[neg_word] -= self.learning_rate * grad * v_target

            print(f"Epoch {epoch} loss {loss:.3f}")
        self.loss = loss
    
    def sigmoid(self, x):
        return 1/(1+np.exp(-x))

    def most_similar(self, word, word2idx, idx2word, top_k=5):
        idx = word2idx[word]
        vec = self.W_in[idx]
        sims=[]
        #cosine similarity calculated for each word
        for i in range(self.vocab_size):
            sim = np.dot(vec, self.W_in[i])/((np.linalg.norm(vec)*np.linalg.norm(self.W_in[i]) +1e-10))
            sims.append((idx2word[i], sim))

        sims.sort(key=lambda x: x[1], reverse=True)
        return sims[1:top_k+1] #we skip the word itself in the output

    def k_means(self, k=10):
        kmeans = KMeans(n_clusters=k)
        #normalization of values
        X = self.W_in / np.linalg.norm(self.W_in, axis=1, keepdims=True)
        clusters = kmeans.fit_predict(X)
        centers = kmeans.cluster_centers_
        return clusters, centers

In [18]:
model = Word2vec(vocab_size = vocab_size)
model.train(pairs)

Epoch 0 loss 2117624.328
Epoch 1 loss 1817765.523
Epoch 2 loss 1724093.194
Epoch 3 loss 1654524.351
Epoch 4 loss 1603597.892


#### Model evaluation
There is no one good way to evaluate word2vec model. First idea is to look for words that are very popular and are good representation of some genre of words. Example of such words choosen by me are: 'time'- appear 737 times, 'film' - 403 times, 'number' - 398 times, game - 379 times, 'music' - 341 times. I encourage to experiment with the words similarities by yourself. Code for finding most popular words can be found in the words exploration section

In [19]:
print(model.most_similar("time", word2idx, idx2word))
print(model.most_similar("film", word2idx, idx2word))
print(model.most_similar("number", word2idx, idx2word))
print(model.most_similar("game", word2idx, idx2word))
print(model.most_similar("music", word2idx, idx2word))

[('ning', np.float64(0.8752613909771984)), ('immortal', np.float64(0.8694825955357474)), ('cars', np.float64(0.8651814389754597)), ('letters', np.float64(0.859698933717823)), ('phrases', np.float64(0.8574686067601204))]
[('soundtrack', np.float64(0.876425722713621)), ('hollywood', np.float64(0.8755617165171232)), ('movie', np.float64(0.8601051200371923)), ('album', np.float64(0.8518892149419772)), ('string', np.float64(0.840378934694094))]
[('runway', np.float64(0.8592479846307262)), ('atop', np.float64(0.8524044816237006)), ('forty', np.float64(0.8493835859657904)), ('rpm', np.float64(0.8483715249595225)), ('gusts', np.float64(0.8334001257863273))]
[('lang', np.float64(0.869324209105837)), ('mood', np.float64(0.8676846802014506)), ('hence', np.float64(0.8552446045749766)), ('replay', np.float64(0.8508861810783042)), ('kept', np.float64(0.8491731600626364))]
[('christmas', np.float64(0.9063003332038235)), ('critic', np.float64(0.898378394921792)), ('digital', np.float64(0.8945990954792

Results are mixed for different words. For 'film' and 'music' results look promising, they are mostly related to the film and music domains. For 'number' results still look decent Those are words that are usually context for 'number', for example 'average number', but it's not bad. For 'time' results are more random, some are related words, but most of them look like random words. For 'game' the words are totally mixed up, 'xlvii' probably is reference to some game name, but other words are not related. 

**Important:** above results are only from the one training and they differ slightly across different trainings. This indicate that model is not trained enough and should be trained more for example by increasing: epochs number, negative samples or dataset size. Showing better training will be limited in this example due to time complexity. In general the truth is that 'film' and 'music' show good results. 'number' and 'game' show decent results and 'time' show not so good resuts.

The next evaluation method is about dividing words into clusters using k-means method and finding center words of those cluster that might possibly describe the cluster

In [20]:
model_kclusters30, model_kcenters30 = model.k_means(k=30)

In [21]:
def kmeans_centers(centers, k):
    for i in range(k):
        center = centers[i]
        sims = []
        for j in range(len(model.W_in)):
            vec = model.W_in[j]
            sim = np.dot(center, vec) / (
                np.linalg.norm(center) * np.linalg.norm(vec) + 1e-10)            
            sims.append((idx2word[j], sim))
        
        sims.sort(key=lambda x: x[1], reverse=True)
        print("Cluster ", i+1, ":")
        print([w for w,_ in sims[:10]])

In [22]:
kmeans_centers(model_kcenters30, 30)

Cluster  1 :
['hate', 'proton', 'reactive', 'authentic', 'mediocre', 'songwriting', 'realize', 'flavor', 'brilliant', 'surroundings']
Cluster  2 :
['separation', 'assert', 'fencing', 'solve', 'procedures', 'mowing', 'exclude', 'priority', 'fulfill', 'operator']
Cluster  3 :
['lynette', 'handlers', 'immortal', 'kongo', 'outlook', 'pavarotti', 'seal', 'therapist', 'hoglan', 'activates']
Cluster  4 :
['mcentire', 'aaron', 'grandson', 'sadistic', 'stripped', 'dunn', 'mcdonald', 'lebron', 'mcgurk', 'atkinson']
Cluster  5 :
['successive', 'prospect', 'rookie', 'upcoming', 'cy', 'nc', 'paralympics', 'wnwbl', 'sting', 'festivals']
Cluster  6 :
['christianity', 'unionists', 'collegiate', 'nineteenth', 'mysorean', 'microlight', 'tudor', 'tennis', 'frontier', 'maharashtra']
Cluster  7 :
['artificial', 'distress', 'shortcomings', 'sinful', 'virtues', 'sympathetic', 'forbids', 'violate', 'spousal', 'prohibition']
Cluster  8 :
['windsor', 'brazoria', 'portion', 'newport', 'ignace', 'mainline', 'naza

Although many of the clusters seem like unorganized group there are few that show that he model is working correctly. For example Cluster 2 with metrics related words: 
['averages', '196', '610', 'kn', '139', 'cu', 'millimeters', '990', '540', '508'].
Cluster 11 has words that look like taken from experiment description (hydrophobic, extracted etc.): ['flawless', 'interactions', 'hydrophobic', 'centric', 'processes', 'modeled', 'extracted', 'alphabet', 'prediction', 'poorer']. Cluster 13 has some geography(maybe even history) related words: ['oradea', 'sustainable', 'greenwich', '16th', 'hellenic', 'montenegrin', '1768', 'stabilization', 'onward', 'neighbouring'].
Cluster 21 contain words that are probably names or surnames: ['worthing', 'stacy', 'johnston', 'remixing', 'mirkin', 'kerr', 'stafford', 'penfold', 'dukes', 'gielgud'].
Those are only some relationships that can be found and I encourage everyone to try and find some more

#### Model summary
Model created here is a proper example of working Word2vec model - skip-gram with negative sampling. Due to time limitation the training was shortened and results might not be satisfactory for everyone, but the model is working correctly and it can be tuned to be very good model. Some key points for future developement migh be:
- Time for training was limited, proper model training should include more data as input, more epochs and probably also more negative samples for each iteration or more number of embedding dimensions.
- Window size that defines size of words neighborhood was set at 2 for this model. This is rather small value, bigger value could catch more complex relationships
- Model inludes many rare words. Focusing only on more popular words could increase model quality, but could also loose some important relations
- Data comes from Wikipedia pages. Although it's not an concern of this example this can cause model to be adjusted only to some specific encyclopedic type of language. Supporting model with training on data from other sources could benefit the model

#### Experiments with model
This is part is not in the scope of main task of this example. However, due to my own curiosity I want to check what would be the model performance in case of focusing only on the most popular words by filtering out the rare words

In [23]:
word_number_checker(words, 1)
word_number_checker(words, 3)
word_number_checker(words, 5)
word_number_checker(words, 10)
word_number_checker(words, 20)
word_number_checker(words, 50)

Appearing more than 1 times: 13570
Appearing more than 3 times: 13570
Appearing more than 5 times: 9307
Appearing more than 10 times: 5593
Appearing more than 20 times: 3174
Appearing more than 50 times: 1302


In [24]:
def filter_by_popularity(words, min_count=1):
    word_counts = Counter(words)
    vocab = [word for word, c in word_counts.items() if c >= min_count]
    vocab_set = set(vocab)
    words = [word for word in words if word in vocab_set]
    vocab_size = len(vocab)
    print(vocab[:20])
    return words, vocab, vocab_size

In [25]:
def model_evaluation(model, word2idx, idx2word):
    print(model.most_similar("time", word2idx, idx2word))
    print(model.most_similar("film", word2idx, idx2word))
    print(model.most_similar("number", word2idx, idx2word))
    print(model.most_similar("game", word2idx, idx2word))
    print(model.most_similar("music", word2idx, idx2word))
    model_kclusters30, model_kcenters30 = model.k_means(k=30)
    kmeans_centers(model_kcenters30, 30)

In [31]:
def model_by_word_popularity_experiment(words, min_count):
    words, vocab, vocab_size = filter_by_popularity(words, min_count)
    word2idx = {w:i for i,w in enumerate(vocab)}
    idx2word = {i:w for w,i in word2idx.items()}
    corpus = [word2idx[w] for w in words]
    pairs = generate_pairs(corpus)
    model = Word2vec(vocab_size=vocab_size)
    model.train(pairs)
    model_evaluation(model, word2idx, idx2word)

In [29]:
model_by_word_popularity_experiment(words, 5)

['valkyria', 'chronicles', 'iii', 'senjō', 'no', '3', 'unk', 'japanese', 'lit', 'of', 'the', 'battlefield', 'commonly', 'referred', 'to', 'as', 'outside', 'japan', 'is', 'a']
Epoch 0 loss 2069226.836
Epoch 1 loss 1823751.328
Epoch 2 loss 1733294.177
Epoch 3 loss 1674572.586
Epoch 4 loss 1634133.993
[('flies', np.float64(0.853149049396349)), ('therapist', np.float64(0.837753452225781)), ('conception', np.float64(0.823668513309056)), ('brandt', np.float64(0.8175536756817813)), ('rise', np.float64(0.8169838288578446))]
[('soundtrack', np.float64(0.897819650802952)), ('script', np.float64(0.8814733855010773)), ('fashion', np.float64(0.8710888958423855)), ('movie', np.float64(0.8695181427064435)), ('tour', np.float64(0.8617006677439419))]
[('intensity', np.float64(0.8883909452132468)), ('peaking', np.float64(0.877106313121569)), ('cyclones', np.float64(0.8671553021032662)), ('atop', np.float64(0.8653395806222537)), ('decade', np.float64(0.8649122256315332))]
[('girls', np.float64(0.85965961

Limiting words only to the ones that appear at least 5 times have made model performing worse, this brings a question is the same happening for limiting words to the ones that appear at least 3 times? 

In [32]:
words = preprocess(dataset)
words = words[:500000] #we need to load words again, cause it is already filtered
model_by_word_popularity_experiment(words, 1)

['valkyria', 'chronicles', 'iii', 'senjō', 'no', '3', 'unk', 'japanese', '戦場のヴァルキュリア3', 'lit', 'of', 'the', 'battlefield', 'commonly', 'referred', 'to', 'as', 'outside', 'japan', 'is']
Epoch 0 loss 2153246.121
Epoch 1 loss 1843349.018
Epoch 2 loss 1742587.526
Epoch 3 loss 1673865.183
Epoch 4 loss 1623822.910
[('dispute', np.float64(0.8581869732034713)), ('conviction', np.float64(0.8529453001378416)), ('speech', np.float64(0.8485270268798428)), ('anthem', np.float64(0.8479872130967246)), ('grid', np.float64(0.8422480185170071))]
[('hollywood', np.float64(0.8730838202884919)), ('guide', np.float64(0.8698266854432575)), ('soundtrack', np.float64(0.8423881572976524)), ('kheops', np.float64(0.8297367661584786)), ('drama', np.float64(0.8259153500991084))]
[('atop', np.float64(0.8488719940360453)), ('twa', np.float64(0.8452730070449175)), ('least', np.float64(0.8317375766922822)), ('rpm', np.float64(0.824664435818439)), ('topped', np.float64(0.8246184799152331))]
[('rally', np.float64(0.86974

If we include every word similarities for choosen words seem comparable with min3 model. However, many words appear as center words in many clusters. This may indicate that words are not differentiated enough. Obviously, this could also mean too many clusters, but 30 clusters doesn't seem like a too big choice for word model, so it's rather problem of including too much words.

Even though I was choosing number 3 as filtering number only by intuition it appears it was pretty good choice